# Exercise 2 — Build a camera gauge, in Python
*Quality Management · Hands-on lab · Università degli Studi di Perugia*

What the [web page](https://lollocappo.github.io/student-hub/materials/quality-management/hands-on/gauge.html) does, step by step. The example photo is synthetic, so the true diameters are known and you can check how good the method is. Run the cells in order with **Shift + Enter**.

It builds on [Exercise 1](https://lollocappo.github.io/student-hub/materials/quality-management/hands-on/camera.html): a colour channel to measure in, a threshold that makes a 1-bit image, the noise of the camera, and averaging.

In [ ]:
# Run this cell first: it downloads the example files from the course repository.
import os, urllib.request
BASE = "https://raw.githubusercontent.com/LolloCappo/student-hub/main/materials/quality-management/hands-on/"

def fetch(path):
    """Download a file of the lab once, and return its local name."""
    name = os.path.basename(path)
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + path, name)
    return name

PHOTO = fetch("img/demo-washers.jpg")    # 11 washers on a backlight, 0.08 mm per pixel

In [ ]:
# Optional — use your own photo of washers instead of the example.
# Remove the "#" in front of the two lines below, run the cell, and pick the file.
# from google.colab import files
# PHOTO = list(files.upload())[0]

## 1. Channel, grey levels and the threshold
Choose the channel where washer and background differ most (Exercise 1, step 3). Pixels darker than the threshold count as washer — a 1-bit image (Exercise 1, step 4). Otsu's method puts the threshold in the valley between the two peaks of the histogram.

In [ ]:
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

CHANNEL = "grey"                       # or "red", "green", "blue"
bgr = cv2.imread(PHOTO)                # OpenCV stores colours as blue, green, red
img = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY) if CHANNEL == "grey" else bgr[..., {"blue": 0, "green": 1, "red": 2}[CHANNEL]]
otsu, _ = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("suggested threshold:", otsu)

# contrast in units of the noise (Exercise 1, step 6); noise from neighbouring background pixels
bg = img > otsu
pairs = bg[:, :-1] & bg[:, 1:]
noise = np.abs(np.diff(img.astype(float), axis=1))[pairs].mean() * np.sqrt(np.pi) / 2
print(f"washers {img[~bg].mean():.0f}, background {img[bg].mean():.0f}: contrast {img[bg].mean() - img[~bg].mean():.0f} grey levels = {(img[bg].mean() - img[~bg].mean()) / noise:.0f} x the noise ({noise:.1f})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].imshow(img, cmap="gray"); ax[0].axis("off")
ax[1].hist(img.ravel(), bins=256, range=(0, 256), color="#2B4D9A")
ax[1].set_yscale("log"); ax[1].axvline(otsu, color="r"); ax[1].set_xlabel("grey level"); ax[1].set_ylabel("pixels (log scale)")
plt.show()

## 2. Find the washers and measure them in pixels
Fill each washer's hole, give every washer a number, and turn **areas** into diameters: Ø = 2√(A/π).

In [ ]:
def measure(img, t):
    """Outer and inner diameter, in pixels, of every washer darker than threshold t."""
    metal = img < t
    filled = ndimage.binary_fill_holes(metal)
    lab, n = ndimage.label(filled)
    idx = np.arange(1, n + 1)
    a_out = ndimage.sum(filled, lab, idx)
    a_in = a_out - ndimage.sum(metal, lab, idx)
    cy, cx = np.array(ndimage.center_of_mass(filled, lab, idx)).T
    edge = np.unique(np.r_[lab[0], lab[-1], lab[:, 0], lab[:, -1]])
    keep = (a_out > 0.0003 * img.size) & ~np.isin(idx, edge)
    df = pd.DataFrame({"cx": cx, "cy": cy, "od_px": 2 * np.sqrt(a_out / np.pi), "id_px": 2 * np.sqrt(a_in / np.pi)})[keep]
    row = (df.cy / df.od_px.median()).round().astype(int)                # reading order: rows, then left to right
    return df.assign(row=row).sort_values(["row", "cx"]).drop(columns="row").reset_index(drop=True)

parts = measure(img, otsu)
print(len(parts), "washers"); parts.round(1)

In [ ]:
plt.figure(figsize=(9, 7)); plt.imshow(img, cmap="gray")
for k, p in parts.iterrows():
    for d in (p.od_px, p.id_px):
        plt.gca().add_patch(plt.Circle((p.cx, p.cy), d / 2, fill=False, color="gold", lw=1.2))
    plt.text(p.cx, p.cy - p.od_px * 0.65, str(k), color="white", ha="center", fontsize=9, bbox=dict(fc="k", alpha=0.6, lw=0))
plt.axis("off"); plt.show()

## 3. Calibrate on the master washer
With the master's **outer** diameter only: one unknown, the scale *k* (mm per pixel).
With its **outer and inner** diameters: two unknowns, the scale and the edge shift *b* caused by the threshold —
`OD_px = OD/k + 2b` and `ID_px = ID/k − 2b`, so `k = (OD + ID) / (OD_px + ID_px)`.

In [ ]:
MASTER = 0                     # index of the master in the picture above
MASTER_OD, MASTER_ID = 15.94, 8.46          # its caliper readings, mm (these are the example's)

def calibrate(parts, both=True):
    m = parts.loc[MASTER]
    if both:
        k = (MASTER_OD + MASTER_ID) / (m.od_px + m.id_px)
        b = (m.od_px - MASTER_OD / k) / 2
    else:
        k, b = MASTER_OD / m.od_px, 0.0
    return parts.assign(od_mm=(parts.od_px - 2 * b) * k, id_mm=(parts.id_px + 2 * b) * k), k, b

res, k, b = calibrate(parts)
print(f"scale {k:.5f} mm per pixel, edge shift {b:+.2f} px")
res[["od_mm", "id_mm"]].round(3)

## 4. How good is it? (the example only)
The example photo was generated with known diameters. Compare.

In [ ]:
import json, re
truth_js = open(fetch("img/demo-washers.js")).read()
truth = json.loads(re.search(r"DEMO_TRUTH = (\{.*\});", truth_js).group(1))
T = pd.DataFrame(truth["parts"])
# match every measured washer to the nearest true one
j = [int(np.argmin(np.hypot(T.cx_px - p.cx, T.cy_px - p.cy))) for _, p in res.iterrows()]
err = pd.DataFrame({"OD error (µm)": (res.od_mm.values - T.od.values[j]) * 1000,
                    "ID error (µm)": (res.id_mm.values - T.id.values[j]) * 1000})
err.round(1).describe().loc[["mean", "std", "min", "max"]]

## 5. Move the threshold
Calibrated on the outer diameter only, the inner diameter follows the threshold. Calibrated on both, it hardly moves.

In [ ]:
PART = 3
rows = []
for t in range(int(otsu) - 30, int(otsu) + 31, 5):
    p = measure(img, t)
    for both in (False, True):
        r, *_ = calibrate(p, both)
        rows.append({"threshold": t, "calibration": "outer + inner" if both else "outer only",
                     "OD": r.od_mm[PART], "ID": r.id_mm[PART]})
sens = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharex=True)
for cal, g in sens.groupby("calibration"):
    ax[0].plot(g.threshold, g.OD, "o-", label=cal); ax[1].plot(g.threshold, g.ID, "o-", label=cal)
ax[0].set_title(f"washer {PART}: outer Ø (mm)"); ax[1].set_title(f"washer {PART}: inner Ø (mm)")
for a in ax: a.set_xlabel("threshold"); a.grid(alpha=0.3); a.legend()
plt.show()

## Questions
As on the web page. One more for Python users: in `measure`, what would change if you measured the diameter between the two outermost pixels instead of from the area — and why is the area better?